
# Onboarding Experiment Investigation

**Assignment:** A/B test investigation of an onboarding flow

### Goal
Investigate whether the new onboarding flow (`treatment`) is genuinely better overall, better for specific user segments, or whether the overall result is affected by segment composition / assignment.

### Required questions
1. Overall naive treatment-vs-control conversion lift.
2. Segment-level conversion rates, sample sizes, and an untrustworthy-looking lift if any.
3. Mix-adjusted overall lift using each segment's share of the total population.
4. Segment with a real, meaningful positive effect, if any.
5. Bonus: treatment/control assignment balance by segment.

The assignment requires `ANSWERS.md` and `answers.json`, and asks for the investigation process and any dead ends checked.



## 0. Setup

**Important:** Put the supplied CSV in the same folder as this notebook.

The notebook below uses the filename:

`experiment_results.csv`

If your file still has its original name `experiment_results (1).csv`, either rename it to `experiment_results.csv` or change `FILE_NAME` in the next cell.


In [1]:

import pandas as pd
import numpy as np
import math
import json
from pathlib import Path
import matplotlib.pyplot as plt

FILE_NAME = "experiment_results.csv"

df = pd.read_csv(FILE_NAME)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())


Dataset shape: (14000, 4)
Columns: ['user_id', 'segment', 'variant', 'converted']


,user_id,segment,variant,converted
0,103792,paid_search,control,0
1,101683,organic,treatment,0
2,107268,app_store,treatment,0
3,108837,paid_search,control,0
4,101617,paid_search,control,0



## 1. Data validation

Before calculating experiment results, check:

- missing values
- duplicate user IDs
- expected variants
- expected segments
- binary conversion values

These checks help make sure the calculations are being performed on the intended data.


In [2]:

print("Missing values:")
display(df.isnull().sum().to_frame("missing_values"))

print("Duplicate user IDs:", df["user_id"].duplicated().sum())

print("\nVariants:")
print(sorted(df["variant"].unique()))

print("\nSegments:")
print(sorted(df["segment"].unique()))

print("\nConverted values:")
print(sorted(df["converted"].unique()))


Missing values:


,missing_values
user_id,0
segment,0
variant,0
converted,0


Duplicate user IDs: 0

Variants:
['control', 'treatment']

Segments:
['app_store', 'influencer', 'organic', 'paid_search', 'referral']

Converted values:
[np.int64(0), np.int64(1)]



### Validation result

For this dataset:

- 14,000 user rows are present.
- There are 4 columns: `user_id`, `segment`, `variant`, and `converted`.
- There are no missing values.
- There are no duplicate `user_id` values.
- Variants are `control` and `treatment`.
- The five segments are `app_store`, `influencer`, `organic`, `paid_search`, and `referral`.
- `converted` is binary: 0/1.

No data-cleaning step was required before the experiment calculations.



# Q1. Overall (naive) treatment vs control

### Formula

**Conversion rate = conversions / users**

**Naive lift = treatment conversion rate − control conversion rate**

The answer is reported in **percentage points**, not relative percent improvement.


In [3]:

overall = (
    df.groupby("variant")
      .agg(
          users=("user_id", "count"),
          conversions=("converted", "sum"),
          conversion_rate=("converted", "mean")
      )
)

overall["conversion_rate_pct"] = overall["conversion_rate"] * 100

control_n = int(overall.loc["control", "users"])
treatment_n = int(overall.loc["treatment", "users"])

control_conversions = int(overall.loc["control", "conversions"])
treatment_conversions = int(overall.loc["treatment", "conversions"])

control_rate = float(overall.loc["control", "conversion_rate"])
treatment_rate = float(overall.loc["treatment", "conversion_rate"])

naive_lift_pp = (treatment_rate - control_rate) * 100

display(overall[["users", "conversions", "conversion_rate_pct"]].round(4))

print(f"Control users: {control_n:,}")
print(f"Treatment users: {treatment_n:,}")
print(f"Control conversion rate: {control_rate*100:.4f}%")
print(f"Treatment conversion rate: {treatment_rate*100:.4f}%")
print(f"Naive lift: {naive_lift_pp:.4f} percentage points")


,users,conversions,conversion_rate_pct
variant,,,
control,7136,1414,19.8150
treatment,6864,1814,26.4277


Control users: 7,136
Treatment users: 6,864
Control conversion rate: 19.8150%
Treatment conversion rate: 26.4277%
Naive lift: 6.6127 percentage points



### Q1 answer

- **Control users:** 7,136
- **Treatment users:** 6,864
- **Control conversion rate:** 19.8150%
- **Treatment conversion rate:** 26.4277%
- **Naive lift:** **+6.61 percentage points**

Calculation:

`26.4277% − 19.8150% = 6.6127 percentage points`



# Q2. Segment-level analysis

Now compare treatment and control **within each segment**.

This is important because the overall result can be misleading if treatment and control contain different proportions of high- and low-converting segments.


In [4]:

segment_results = []

for segment in sorted(df["segment"].unique()):
    segment_data = df[df["segment"] == segment]

    control = segment_data[segment_data["variant"] == "control"]
    treatment = segment_data[segment_data["variant"] == "treatment"]

    control_n_seg = len(control)
    treatment_n_seg = len(treatment)

    control_rate_seg = control["converted"].mean()
    treatment_rate_seg = treatment["converted"].mean()

    lift_pp_seg = (treatment_rate_seg - control_rate_seg) * 100

    total_seg = len(segment_data)
    population_share = total_seg / len(df)

    control_share = control_n_seg / total_seg
    treatment_share = treatment_n_seg / total_seg

    segment_results.append({
        "segment": segment,
        "total_users": total_seg,
        "population_share_pct": population_share * 100,
        "control_n": control_n_seg,
        "control_conversion_pct": control_rate_seg * 100,
        "treatment_n": treatment_n_seg,
        "treatment_conversion_pct": treatment_rate_seg * 100,
        "lift_pp": lift_pp_seg,
        "control_share_pct": control_share * 100,
        "treatment_share_pct": treatment_share * 100
    })

segment_df = pd.DataFrame(segment_results)

display(segment_df.round(4))


,segment,total_users,population_share_pct,control_n,control_conversion_pct,treatment_n,treatment_conversion_pct,lift_pp,control_share_pct,treatment_share_pct
0,app_store,1885,13.4643,925,8.7568,960,20.0000,11.2432,49.0716,50.9284
1,influencer,250,1.7857,119,23.5294,131,16.7939,-6.7355,47.6000,52.4000
2,organic,4215,30.1071,1298,35.2851,2917,35.0703,-0.2148,30.7948,69.2052
3,paid_search,4812,34.3714,3353,15.1804,1459,14.3934,-0.7870,69.6800,30.3200
4,referral,2838,20.2714,1441,23.4559,1397,26.2706,2.8146,50.7752,49.2248


In [5]:

# Exact conversions and rates for each segment/variant

detail_rows = []

for segment in sorted(df["segment"].unique()):
    for variant in ["control", "treatment"]:
        sub = df[(df["segment"] == segment) & (df["variant"] == variant)]

        detail_rows.append({
            "segment": segment,
            "variant": variant,
            "users": len(sub),
            "conversions": int(sub["converted"].sum()),
            "conversion_rate_pct": sub["converted"].mean() * 100
        })

segment_detail = pd.DataFrame(detail_rows)

display(segment_detail.round(4))


,segment,variant,users,conversions,conversion_rate_pct
0,app_store,control,925,81,8.7568
1,app_store,treatment,960,192,20.0000
2,influencer,control,119,28,23.5294
3,influencer,treatment,131,22,16.7939
4,organic,control,1298,458,35.2851
5,organic,treatment,2917,1023,35.0703
6,paid_search,control,3353,509,15.1804
7,paid_search,treatment,1459,210,14.3934
8,referral,control,1441,338,23.4559
9,referral,treatment,1397,367,26.2706



### Q2 interpretation

The segment results are:

| Segment | Control N | Control CR | Treatment N | Treatment CR | Lift |
|---|---:|---:|---:|---:|---:|
| app_store | 925 | 8.76% | 960 | 20.00% | **+11.24 pp** |
| influencer | 119 | 23.53% | 131 | 16.79% | **−6.74 pp** |
| organic | 1,298 | 35.29% | 2,917 | 35.07% | **−0.21 pp** |
| paid_search | 3,353 | 15.18% | 1,459 | 14.39% | **−0.79 pp** |
| referral | 1,441 | 23.46% | 1,397 | 26.27% | **+2.81 pp** |

There is **no segment that simultaneously has an impressive positive lift and a small sample**. `influencer` has the smallest sample, but its lift is negative. `app_store` has the largest positive lift, but its sample is large and its treatment/control assignment is close to 50/50.

Therefore, for the JSON field `q2_untrustworthy_segment`, the answer is **`none`**.



# Q3. Mix-adjusted overall lift

The assignment specifically asks for:

> each segment's treatment-vs-control lift, weighted by that segment's share of the total user population.

So:

**Mix-adjusted lift = Σ(segment population share × segment lift)**

The weight is **not** based on the number of treatment users.


In [6]:

segment_df["weighted_lift_pp"] = (
    segment_df["population_share_pct"] / 100
    * segment_df["lift_pp"]
)

for _, row in segment_df.iterrows():
    print(
        f"{row['segment']}: "
        f"{row['population_share_pct']:.4f}% × "
        f"{row['lift_pp']:.4f} pp = "
        f"{row['weighted_lift_pp']:.4f} pp"
    )

mix_adjusted_lift_pp = round(
    segment_df["weighted_lift_pp"].sum(),
    2
)

print(f"\nMix-adjusted overall lift: {mix_adjusted_lift_pp:.2f} percentage points")
print(f"Naive overall lift: {naive_lift_pp:.2f} percentage points")
print(f"Difference: {naive_lift_pp - mix_adjusted_lift_pp:.2f} percentage points")


app_store: 13.4643% × 11.2432 pp = 1.5138 pp
influencer: 1.7857% × -6.7355 pp = -0.1203 pp
organic: 30.1071% × -0.2148 pp = -0.0647 pp
paid_search: 34.3714% × -0.7870 pp = -0.2705 pp
referral: 20.2714% × 2.8146 pp = 0.5706 pp

Mix-adjusted overall lift: 1.63 percentage points
Naive overall lift: 6.61 percentage points
Difference: 4.98 percentage points



### Q3 answer

The weighted contributions are:

- `app_store`: **+1.5138 pp**
- `influencer`: **−0.1203 pp**
- `organic`: **−0.0647 pp**
- `paid_search`: **−0.2705 pp**
- `referral`: **+0.5706 pp**

Therefore:

**Mix-adjusted overall lift = +1.63 percentage points**

The result differs substantially from the **+6.61 pp naive lift** because treatment and control do not have the same segment composition. Treatment contains far more `organic` users, while control contains far more `paid_search` users. These segments have different baseline conversion rates, so the raw overall comparison is affected by this composition imbalance.



# Q4. Is there a real, meaningful positive segment effect?

For this question, do **not** simply select the largest raw lift. Consider:

1. Effect size.
2. Treatment sample size.
3. Control sample size.
4. Treatment/control balance.
5. Segment population share.
6. Whether the result is stable under an uncertainty check.

The assignment does not require a significance test, but the following two-proportion check is included as **supplementary evidence**, not as a replacement for the requested calculations.


In [7]:

def two_prop_stats(x_t, n_t, x_c, n_c):
    p_t = x_t / n_t
    p_c = x_c / n_c
    diff = p_t - p_c

    se = math.sqrt(
        p_t * (1 - p_t) / n_t +
        p_c * (1 - p_c) / n_c
    )

    ci_low = diff - 1.96 * se
    ci_high = diff + 1.96 * se

    p_pool = (x_t + x_c) / (n_t + n_c)

    se_null = math.sqrt(
        p_pool * (1 - p_pool) *
        (1 / n_t + 1 / n_c)
    )

    z = diff / se_null if se_null else np.nan
    p_value = math.erfc(abs(z) / math.sqrt(2))

    return diff * 100, ci_low * 100, ci_high * 100, p_value


stats_rows = []

for segment in sorted(df["segment"].unique()):
    c = df[(df["segment"] == segment) & (df["variant"] == "control")]
    t = df[(df["segment"] == segment) & (df["variant"] == "treatment")]

    lift, ci_low, ci_high, p_value = two_prop_stats(
        int(t["converted"].sum()),
        len(t),
        int(c["converted"].sum()),
        len(c)
    )

    stats_rows.append({
        "segment": segment,
        "lift_pp": lift,
        "95% CI low (pp)": ci_low,
        "95% CI high (pp)": ci_high,
        "p_value": p_value
    })

stats_df = pd.DataFrame(stats_rows)

display(stats_df.round(4))


,segment,lift_pp,95% CI low (pp),95% CI high (pp),p_value
0,app_store,11.2432,8.1254,14.3611,0.0000
1,influencer,-6.7355,-16.6886,3.2176,0.1836
2,organic,-0.2148,-3.3384,2.9089,0.8927
3,paid_search,-0.7870,-2.9595,1.3854,0.4815
4,referral,2.8146,-0.3654,5.9947,0.0828



### Q4 answer: `app_store`

`app_store` shows:

- Control: **925 users**, **8.76%** conversion.
- Treatment: **960 users**, **20.00%** conversion.
- Lift: **+11.24 percentage points**.
- Population share: **13.46%**.
- Assignment: **49.07% control / 50.93% treatment**.
- Supplementary 95% CI: approximately **+8.13 to +14.36 pp**.
- Supplementary p-value: **< 0.0001**.

This combination of a large observed effect, substantial sample size, near-even allocation, and uncertainty interval entirely above zero makes `app_store` the clearest segment-level positive effect in this dataset.

This does **not** mean the treatment is better for every segment: `organic` and `paid_search` have slightly negative within-segment lifts, while `influencer` is also negative.



# Q5. Bonus — assignment balance

We need the fraction of each segment assigned to treatment versus control.


In [8]:

assignment_counts = pd.crosstab(
    df["segment"],
    df["variant"]
)

assignment_pct = (
    pd.crosstab(
        df["segment"],
        df["variant"],
        normalize="index"
    ) * 100
)

print("Assignment counts:")
display(assignment_counts)

print("Assignment percentages within each segment:")
display(assignment_pct.round(2))

print("Overall assignment:")
display((df["variant"].value_counts(normalize=True) * 100).round(2))


Assignment counts:


variant,control,treatment
segment,,
app_store,925,960
influencer,119,131
organic,1298,2917
paid_search,3353,1459
referral,1441,1397


Assignment percentages within each segment:


variant,control,treatment
segment,,
app_store,49.07,50.93
influencer,47.60,52.40
organic,30.79,69.21
paid_search,69.68,30.32
referral,50.78,49.22


Overall assignment:


variant
control      50.97
treatment    49.03
Name: proportion, dtype: float64


### Q5 answer

The major allocation imbalances are:

- `organic`: **30.79% control / 69.21% treatment**
- `paid_search`: **69.68% control / 30.32% treatment**

The other segments are much closer to an even split.

Interestingly, the **overall** assignment is still close to 50/50:

- Control: **50.97%**
- Treatment: **49.03%**

Therefore, checking only the overall treatment/control counts would miss the important segment-level allocation imbalance.



# Investigation process and dead ends

These are the points to include in the final submission:

1. Loaded and inspected the experiment CSV.
2. Checked missing values; none were found.
3. Checked duplicate user IDs; none were found.
4. Verified the expected variants, segments, and binary conversion values.
5. Calculated the overall naive treatment/control conversion difference.
6. Calculated conversion rates and lifts separately for every segment.
7. Checked whether a very large segment lift was supported by sufficient observations rather than selecting the largest percentage blindly.
8. Calculated the population-share-weighted mix-adjusted lift.
9. Checked treatment/control assignment proportions separately within every segment.
10. **Dead end:** selecting the segment solely by the largest raw lift was rejected because it ignores sample size and treatment/control allocation.
11. Added a supplementary two-proportion uncertainty check to help distinguish a stable effect from a noisy estimate.



# Final answers for `answers.json`

```json
{
  "q1_naive_lift_pp": 6.61,
  "q1_n_control": 7136,
  "q1_n_treatment": 6864,
  "q2_untrustworthy_segment": "none",
  "q3_mix_adjusted_lift_pp": 1.63,
  "q4_real_effect_segment": "app_store"
}
```

## Final conclusion

The naive overall treatment lift is **+6.61 percentage points**, but the mix-adjusted lift is only **+1.63 percentage points**.

The gap is explained by substantial segment-level treatment/control allocation imbalance, especially `organic` and `paid_search`.

Within segments, the new flow is not uniformly better. `organic` and `paid_search` show small negative lifts, `influencer` shows a negative lift, and `referral` shows a smaller positive lift. The clearest positive effect is in **`app_store`**, where conversion rises from **8.76% to 20.00% (+11.24 pp)** with nearly balanced assignment and a substantial sample.

**Key takeaway:** the overall topline should not be interpreted without looking at segment-level effects and assignment composition.


In [9]:

# Final machine-checkable answers

answers = {
    "q1_naive_lift_pp": round(naive_lift_pp, 2),
    "q1_n_control": control_n,
    "q1_n_treatment": treatment_n,
    "q2_untrustworthy_segment": "none",
    "q3_mix_adjusted_lift_pp": round(mix_adjusted_lift_pp, 2),
    "q4_real_effect_segment": "app_store"
}

print(json.dumps(answers, indent=2))


{
  "q1_naive_lift_pp": 6.61,
  "q1_n_control": 7136,
  "q1_n_treatment": 6864,
  "q2_untrustworthy_segment": "none",
  "q3_mix_adjusted_lift_pp": 1.63,
  "q4_real_effect_segment": "app_store"
}
